In [0]:
%pip install pendulum

In [0]:
import json
from pathlib import Path
import pandas as pd
import pendulum

In [0]:
yesterday = pendulum.now("America/Bogota").subtract(days=1)

path = f"/Volumes/workspace/raw/uses/{yesterday.year}/{yesterday.month:02d}/{yesterday.day:02d}"

df_raw = spark.read.json(path)

In [0]:
print(df_raw)
print (path)

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.bronze
COMMENT 'Capa Bronze: datos crudos procesados'

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.uses (
  id BIGINT,
  fechaCreacion TIMESTAMP,
  validator_id BIGINT,
  tsn BIGINT,
  bus_id BIGINT,
  line_id BIGINT,
  applic_id BIGINT,
  type_use BIGINT,
  fee BIGINT,
  uid_target BIGINT,
  date TIMESTAMP,
  flee_control STRING,
  state BIGINT,
  state_description STRING,
  tp_Id BIGINT,
  purseavalue DECIMAL(7,2),
  pursebvalue DECIMAL(7,2),
  cut_id INT,
  errorcode BIGINT
)
USING DELTA
""")

In [0]:
#Sacamos los datos del json data.transactions es un array y lista los registros
#Explore convierte la lista a filas
from pyspark.sql.functions import explode, col

df = df_raw.select(
    explode(col("data.transactions")).alias("trx")
).select("trx.*")

In [0]:
# ajustamos los campos del json al formato que tiene la tabla
df_cast = df.select(
    col("id").cast("bigint"),
    col("fechaCreacion").cast("timestamp"),
    col("validator_id").cast("bigint"),
    col("tsn").cast("bigint"),
    col("bus_id").cast("bigint"),
    col("line_id").cast("bigint"),
    col("applic_id").cast("bigint"),
    col("type_use").cast("bigint"),
    col("fee").cast("bigint"),
    col("uid_target").cast("bigint"),
    col("date").cast("timestamp"),
    col("flee_control").cast("string"),
    col("state").cast("bigint"),
    col("state_description").cast("string"),
    col("tp_Id").alias("tp_id").cast("bigint"),
    col("purseavalue").cast("decimal(7,2)"),
    col("pursebvalue").cast("decimal(7,2)"),
    col("cut_id").cast("int"),
    col("errorcode").cast("bigint")
)

In [0]:
print(df_cast)

In [0]:
df_cast.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.uses")